In [5]:
from pathlib import Path

import pandas as pd

from core.utils.seed import set_seed

set_seed()

TEST_ALLOCATION = 0.2
SPLIT_OUT_PATH = Path("../../data/out/splits/single_token_entropy/mmlu/qwen_3b/")
CHUNK_CNT = 6

entropy_col = "entropy_value"

df = pd.read_parquet("../../data/out/single_token_entropy/mmlu_qwen_3b.parquet")

In [6]:
from pandas import DataFrame

filtered_df = df[df[entropy_col].notna()]

sorted_df = filtered_df.sort_values(entropy_col, ascending=True)

chunk_len = len(sorted_df) // CHUNK_CNT

chunks: list[DataFrame] = []
for i in range(CHUNK_CNT):
    start_idx = i * chunk_len
    # Python (and pandas for that matter) is OK with end index to be out of bounds
    end_idx = start_idx + chunk_len
    chunk = sorted_df.iloc[start_idx:end_idx]
    chunk.reset_index(drop=True, inplace=True)
    chunks.append(chunk)

In [7]:
print("Chunks: ", len(chunks))

for chunk in chunks:
    print("Chunk len: ", len(chunk))
    print("Chunk sample")
    print(chunk.head()[entropy_col])
    # check for missing values in the entropy column
    na_count = chunk[entropy_col].isna().sum()
    print("NA count:", na_count)
    assert na_count == 0, "Found NA values in chunk"

Chunks:  6
Chunk len:  2005
Chunk sample
0    2.042132e-09
1    2.780812e-09
2    2.787101e-09
3    4.184975e-09
4    4.319797e-09
Name: entropy_value, dtype: float64
NA count: 0
Chunk len:  2005
Chunk sample
0    0.000062
1    0.000063
2    0.000063
3    0.000063
4    0.000063
Name: entropy_value, dtype: float64
NA count: 0
Chunk len:  2005
Chunk sample
0    0.004533
1    0.004534
2    0.004538
3    0.004544
4    0.004553
Name: entropy_value, dtype: float64
NA count: 0
Chunk len:  2005
Chunk sample
0    0.082552
1    0.082575
2    0.082623
3    0.082634
4    0.082653
Name: entropy_value, dtype: float64
NA count: 0
Chunk len:  2005
Chunk sample
0    0.392897
1    0.393025
2    0.393167
3    0.393354
4    0.393358
Name: entropy_value, dtype: float64
NA count: 0
Chunk len:  2005
Chunk sample
0    0.793556
1    0.793746
2    0.793916
3    0.794349
4    0.794569
Name: entropy_value, dtype: float64
NA count: 0


In [8]:
from core.utils.splitter import split_chunk_into_train_test

for chunk_i, chunk in enumerate(chunks):
    print("Chunk : ", chunk_i)

    train_df, test_df = split_chunk_into_train_test(chunk, TEST_ALLOCATION)
    print("Train len: ", len(train_df))
    print("Test len: ", len(test_df))

    SPLIT_OUT_PATH.mkdir(parents=True, exist_ok=True)

    test_df.to_parquet(str(SPLIT_OUT_PATH.joinpath(f"group{chunk_i}_test.parquet").resolve()), index=False)
    train_df.to_parquet(str(SPLIT_OUT_PATH.joinpath(f"group{chunk_i}_train.parquet").resolve()), index=False)

Chunk :  0
Train len:  1604
Test len:  401
Chunk :  1
Train len:  1604
Test len:  401
Chunk :  2
Train len:  1604
Test len:  401
Chunk :  3
Train len:  1604
Test len:  401
Chunk :  4
Train len:  1604
Test len:  401
Chunk :  5
Train len:  1604
Test len:  401
